# Model Regularization with `PyTorch` and Hyperparameter Tuning with `Optuna` (MNIST dataset)

This lab will guide you through:
*  Training, and evaluating a neural network for multiclass classification on the MNIST dataset using `PyTorch`.
* Regularizing a model to prevent overfitting using `PyTorch`
* Tuning hyperparameters with `Optuna`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter
from torchvision import datasets, transforms

import seaborn as sns
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    mean_absolute_error,
    mean_squared_error
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

# Data Preparation

## 1. Load MNIST dataset

`transform` is a computer vision method from `torchvision` that is used to apply dynamically sequentially several transformation to image data.

`mean` and `std` have to be computed first if we want to normalize data

In [ ]:
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)

mean = (train_dataset.data.float() / 255.0).mean()
std = (train_dataset.data.float() / 255.0).std()

print(f"Mean: {mean.item():.4f}, Std: {std.item():.4f}")

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)) # This Normalization is good but not mandatory. Here we are obliged to provide mean and standard deviation of the dataset, to perform Z-normalization
])

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

## 2. Split train, val, test datasets

We are going to use a complete training setup with 3 splits.

In [ ]:
# train_size = int(0.8 * len(train_dataset))
# val_size = len(train_dataset) - train_size
# train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])

However, for the purpose of the course, we will take a subset of the complete dataset (~10%), to accelerate trainings

We should not subsample test dataset (evaluation costs less than training)

In [ ]:
train_dataset, val_dataset, _ = random_split(train_dataset, [5000, 1000, len(train_dataset) - 6000])

## 3. Dataloaders

In [ ]:
train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)

In [ ]:
print("MNIST dataset sizes (after split):")
print(f"Train subset size: {len(train_dataset)}")
print(f"Validation subset size: {len(val_dataset)}")
print(f"Test size: {len(test_dataset)}")

print("\nMNIST Data shape:")
print(f"X shape: {train_dataset.dataset.data[0].shape}")
print(f"Y shape: {train_dataset.dataset.targets[0].shape}")

## 4. Print out some data

In [ ]:
examples = iter(train_loader)
example_data, example_targets = next(examples)
fig, axes = plt.subplots(1, 6, figsize=(12, 3))
for i in range(6):
    axes[i].imshow(example_data[i][0], cmap="gray")
    axes[i].set_title(f"Label: {example_targets[i]}")
    axes[i].axis("off")
plt.show()

# Model Definition, evaluation and training

## 1. Model Definition

### example 1: hard coded

For the purpose of the class exercise, here is a basic definition of a possible model. Let's put dropout as parameter, to tune its value later:

In [ ]:
import torch.nn.functional as F

class MNISTNet(nn.Module):
    def __init__(self, dropout=0.0):
        super().__init__()
        self.flat = nn.Flatten()
        self.fc1 = nn.Linear(28 * 28, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        self.out = nn.Linear(64, 10)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        x = self.flat(x)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = F.relu(self.fc3(x))
        x = self.dropout(x)
        x = self.out(x)
        return x

model = MNISTNet().to(device)
model

### example 2: with Sequential

Here is the same model using `nn.Sequential` which allows to reduce the code

In [ ]:
class MNISTNet(nn.Module):
  def __init__(self, dropout=0.0):
    super().__init__()
    self.flatten = nn.Flatten()
    self.classifier = nn.Sequential(
        nn.Linear(28 * 28, 256),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(256, 128),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(128, 64),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(64, 10)
    )

  def forward(self, x):
    x = self.flatten(x)
    x = self.classifier(x)
    return x

### option 3: with some modularity

Here another version, in a customizable way, so that it gives the option to tune hypermarameters based on architecture.
* number of hidden layers (except the first one)
* number of neurons per layer


In [ ]:
# import torch.nn.functional as F

# class MNISTNet(nn.Module):
#     def __init__(self, nb_hidden=2, nb_neurons=128, dropout=0.0):
#         super().__init__()
#         self.flat = nn.Flatten()
#         self.fc1 = nn.Linear(28 * 28, nb_neurons)
#         self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
#         self.hidden_layers = nn.ModuleList()
#         for _ in range(nb_hidden):
#             self.hidden_layers.append(nn.Linear(nb_neurons, nb_neurons))
#         self.out = nn.Linear(nb_neurons, 10)

#     def forward(self, x):
#         x = self.flat(x)
#         x = F.relu(self.fc1(x))
#         x = self.dropout(x)
#         for layer in self.hidden_layers:
#             x = F.relu(layer(x))
#             x = self.dropout(x)
#         x = self.out(x)
#         return x

# model = MNISTNet().to(device)
# model

## 2. Evaluation method

In [ ]:
def evaluate_multiclass(model, loader, criterion):
    model.eval()
    losses, trues, preds = [], [], []

    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            outputs = model(x_batch)
            loss = criterion(outputs, y_batch)
            losses.append(loss.item() * len(x_batch))
            batch_preds = outputs.argmax(dim=1).cpu().numpy()
            preds.extend(batch_preds)
            trues.extend(y_batch.cpu().numpy())

    avg_loss = np.sum(losses) / len(loader.dataset)
    preds = np.array(preds)
    trues = np.array(trues)

    correct = (preds == trues).sum().item()
    accuracy = correct / len(trues)

    return avg_loss, accuracy, preds, trues

## 3. Training method

In [ ]:
def model_train_and_log(model, train_loader, val_loader, epochs, device, criterion, optimizer, writer):
  model = model.to(device)
  for epoch in tqdm(range(epochs)):
      model.train()
      running_loss = 0
      for x_batch, y_batch in train_loader:
          x_batch, y_batch = x_batch.to(device), y_batch.to(device)
          optimizer.zero_grad()
          outputs = model(x_batch)
          loss = criterion(outputs, y_batch)
          loss.backward()
          optimizer.step()
          running_loss += loss.item() * len(x_batch)

      avg_train_loss = running_loss / len(train_loader.dataset)
      avg_val_loss, val_acc, _, _ = evaluate_multiclass(model, val_loader, criterion)

      writer.add_scalar("Loss/train", avg_train_loss, epoch)
      writer.add_scalar("Loss/validation", avg_val_loss, epoch)
      writer.add_scalar("Accuracy/validation", val_acc, epoch)
  writer.close()

In [ ]:
model = MNISTNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
writer = SummaryWriter(log_dir="./runs/mnist_classification")

model_train_and_log(model, train_loader, val_loader, 30, device, criterion, optimizer, writer)

# Tensorboard monitoring

In [ ]:
# On Jupyter Notebook or vscode:
# tensorboard --logdir=runs  # then go to --> http://localhost:6006

# on Google colab:
%load_ext tensorboard
%tensorboard --logdir "./runs"

### Post training evaluation (on test dataset)

In [ ]:
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
)

test_loss, test_acc, test_preds, test_trues = evaluate_multiclass(model, test_loader, criterion)

cm = confusion_matrix(test_trues, test_preds)

plt.figure(figsize=(8,8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
print(classification_report(test_trues, test_preds))

### Misclassified Examples

In [ ]:
misclassified = np.where(test_preds != test_trues)[0]
fig, axes = plt.subplots(1, 6, figsize=(12, 3))
for i, idx in enumerate(misclassified[:6]):
    img, label = test_dataset[idx]
    axes[i].imshow(img[0], cmap="gray")
    axes[i].set_title(f"True: {label}, Pred: {test_preds[idx]}")
    axes[i].axis("off")
plt.show()

## 8. Hyperparameter Tuning with Optuna/Optimum


For this tuning phase, we will train model mutilple times using `Optuna`, trying various settings for arameters to tune. Each time we just want to extract validation loss. That's why we don't need to collect metrics every epochs and log it to Tensorboard. Thus, let's define a simpler method for model training.

>Note: we will report a reasonable number of epochs that we defined above, using early stopping.

In [ ]:
!pip install optuna

In [ ]:
def model_train(model, train_loader, val_loader, epochs, device, criterion, optimizer):
  model = model.to(device)
  for epoch in tqdm(range(epochs)):
      model.train()
      for x_batch, y_batch in train_loader:
          x_batch, y_batch = x_batch.to(device), y_batch.to(device)
          preds = model(x_batch)
          loss = criterion(preds, y_batch)
          optimizer.zero_grad()
          loss.backward()
          optimizer.step()
      avg_val_loss, _, _, _ = evaluate_multiclass(model, val_loader, criterion)
  return avg_val_loss

Part of code is commented. Uncomment it if you use the modular version of the model

In [ ]:
import optuna

NUM_EPOCHS = 10

def objective(trial, num_epochs=NUM_EPOCHS):
    # Suggest some hyperparameters to tune
    # nb_hidden = trial.suggest_int("nb_hidden", 1, 3)
    # nb_neurons = trial.suggest_int("nb_neurons", 64, 128, step=64)
    dropout = trial.suggest_float("dropout", 0.0, 0.3)
    learning_rate = trial.suggest_float('lr', 1e-5, 1e-3)
    l2 = trial.suggest_float("l2", 0.0, 1e-3)

    # Create the model with suggested hyperparameters
    # model = MNISTNet(nb_hidden=nb_hidden, nb_neurons=128, dropout=dropout).to(device)
    model = MNISTNet(dropout=dropout).to(device)

    # Define criterion and optimizer with suggested hyperparameters
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=l2)

    # Train the model
    avg_val_loss = model_train(
        model,
        train_loader,
        val_loader,
        epochs=num_epochs,
        device=device,
        criterion=criterion,
        optimizer=optimizer
      )

    return avg_val_loss

In [ ]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=5)

print(f"Best value: {study.best_value} (params: {study.best_params})")

In [ ]:
study.best_value

In [ ]:
study.best_params

In [ ]:
optuna.visualization.plot_optimization_history(study)

In [ ]:
optuna.visualization.plot_slice(study)

In [ ]:
optuna.visualization.plot_param_importances(study)

## Use best setup to train the model

In [ ]:
study.best_params

In [ ]:
writer = SummaryWriter(log_dir="./runs/mnist_classification_tuned")

model = MNISTNet(
    # nb_hidden=study.best_params["nb_hidden"],
    dropout=study.best_params["dropout"]
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=study.best_params["lr"],
    weight_decay=study.best_params["l2"]
)

avg_val_loss = model_train_and_log(
    model,
    train_loader,
    val_loader,
    epochs=NUM_EPOCHS,
    device=device,
    criterion=criterion,
    optimizer=optimizer,
    writer=writer
  )

In [ ]:
# Save and load model
torch.save(model.state_dict(), 'mnist_model_tuned.pth')

# Load model example
# loaded_model = MNISTNet(nb_hidden=study.best_params["nb_hidden"]).to(device) # have to know how many layers, not practical..
loaded_model = MNISTNet().to(device)
loaded_model.load_state_dict(torch.load('mnist_model_tuned.pth'))
loaded_model